# 3. Predict — simulated CRM data (`data/crm_simulated/`)

Same steps as `python -m src.predict --output reports/crm/simulated/final`, using the functions in `src/predict.py`.
Uses the frozen calibrated model saved by `1_train.ipynb`. Open deals are never labelled Lost.

Scores on simulated deals illustrate the workflow only.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import tempfile
import pandas as pd
from src import predict, train
from src.outputs import Outputs


## Settings

Must match the settings used in `1_train.ipynb`.

In [ ]:
DATA_DIR = ROOT / "data/crm_simulated"   # 100,000 simulated deals
QUICK = False            # True: tiny budgets, smoke test only (writes a *smoke* folder)
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load the scoring bundle

In [ ]:
out = Outputs(OUTPUT)
scoring = predict.load(out)
print("Model:", scoring["model"].name)
print("Decision threshold (calibrated):", round(scoring["decision_threshold"], 4))
print("Features:", len(scoring["features"]), "| data:", out.source_data())

## 2. Score the open Engaging deals

Writes `predictions/open_deal_predictions.csv`: P(Won), P(Lost), predicted class, a heuristic priority band,
the inputs that raise or lower each score, and a warning when the account is missing.

In [ ]:
scores = predict.run(OUTPUT)
pd.Series({
    "scored deals": len(scores),
    "predicted Won": int(scores.predicted_outcome.eq("Won").sum()),
    "High / Medium / Low": "/".join(str(int(scores.priority.eq(p).sum())) for p in ["High", "Medium", "Low"]),
    "missing account (warning)": int(scores.account_missing.sum()),
    "mean P(Won)": round(scores.win_probability.mean(), 4),
})

In [ ]:
scores[["opportunity_id", "product", "sales_agent", "win_probability", "priority",
        "positive_factors", "negative_factors", "input_warning"]].head(10)

## 3. Score new deals from a CSV

New deals use the `sales_pipeline.csv` format: `opportunity_id, sales_agent, product, account, engage_date`.
This example takes five Engaging deals from the data as stand-ins; replace `new_deals` with your own file.

In [ ]:
raw = pd.read_csv(out.source_data() / "sales_pipeline.csv")
new_deals = raw.loc[raw.deal_stage.eq("Engaging") & raw.account.notna(),
                    ["opportunity_id", "sales_agent", "product", "account", "engage_date"]].head(5)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "new_deals.csv"
    new_deals.to_csv(path, index=False)
    new_scores = predict.score_file(path, OUTPUT, Path(folder) / "new_deal_scores.csv")
new_scores[["opportunity_id", "win_probability", "predicted_outcome", "priority", "positive_factors"]]